# 03 — Benchmark: BTLA-response TF prioritisation (GREmLN masked-prior vs GENIE3 masked-raw)

Definitive head-to-head. Both models are restricted to the **common GREmLN∩GENIE3 TF universe**
and scored against the **BTLA_TCR_vs_TCR 4 h panel (249 DEGs)**:

* **GREmLN** — CSLS (k=10): a TF's score = number of BTLA seeds among its top-100 embedding
  neighbours (dense rank).
* **GENIE3** — summed **outgoing** TF→BTLA-DEG edge weight (dense rank).

**Primary nomination comparison = seed-EXCLUDED** (BTLA panel genes removed → candidate
regulators). A **seed-INCLUSIVE** ranking is reported separately as biological context, with panel
genes flagged (★) and never called discoveries.

Candidates are then examined with three **orthogonal, non-predictive** validators:
1. **CD4 CRISPRi Perturb-seq** (primary; Stim8hr) — functional target-response;
2. **Paperclip** literature (identical template);
3. **BTLA multi-omics**, split into **independent orthogonal** (protein/PTM/co-IP) vs
   **derived/contextual** (transcript DE, TF/kinase activity, BIONIC).

Held-out seed-TF recovery is **supplementary only** (pending a leakage/asymmetry audit) and does
**not** enter the verdict. Requires the outputs of notebooks 01 (GENIE3 graph) and 02 (GREmLN
embeddings); set `BTLA_BENCH_DATA_ROOT` to where those + the BTLA panel/Perturb-seq live.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import hypergeom, rankdata, spearmanr
from statsmodels.stats.multitest import multipletests

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "scripts" / "bench_utils.py").exists()), _here.parent)
sys.path.insert(0, str(_root / "scripts"))
import bench_utils as bu
import yaml

CFG = yaml.safe_load((bu.repo_root() / "config" / "benchmark_config.yaml").read_text())
# ---- locked parameters (must match notebooks 01/02) ----
CSLS_K, TOPN = 10, 100
DE_PADJ = 0.05                 # CRISPRi knockdown target-response threshold (adj p only)
STIM_PRIMARY, STIM_SENS = "Stim8hr", "Stim48hr"
RANDOM_SEED = 42
CRISPRI_MARGIN = CFG["decision_rule"]["crispri_margin"]   # decision-rule margin (from config)
N_MASKED_EXPR_GENES = int(CFG["genie3"]["n_hvg"])         # masked expression universe (HVGs)

OUT = bu.repo_root() / "results"
TAB, FIG, REG = OUT / "tables", OUT / "figures", OUT / "model_registry"
for d in (TAB, FIG, REG):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", bu.data_root())

## 1. BTLA panel (assert 249), TF list, embeddings, GENIE3 graph

In [ ]:
seeds, seed_dir, seeds_df = bu.load_seeds()
assert len(seeds) == 249, f"BTLA_TCR_vs_TCR panel must be exactly 249 genes, got {len(seeds)}"
tfs = bu.load_tfs()
genes, X = bu.load_gremln_embeddings()
edges = bu.load_edges()
n_up = (seeds_df['direction'] == 'up').sum(); n_down = (seeds_df['direction'] == 'down').sum()
print(f"BTLA panel: {len(seeds)} genes (up={n_up}, down={n_down}) | TF list: {len(tfs)}")
print(f"GREmLN embedded genes: {len(genes)} | GENIE3 masked edges: {len(edges)}")

## 2. CSLS similarity and the common (scorable-by-both) TF universe

Cosine → CSLS (Conneau et al. 2018) corrects hubness: `CSLS(i,j) = 2·cos(i,j) − r_k(i) − r_k(j)`
with `r_k` the mean cosine to the k nearest neighbours.

In [ ]:
def cosine_similarity_matrix(X):
    n = np.linalg.norm(X, axis=1, keepdims=True); n = np.where(n == 0, 1.0, n)
    Xn = X / n; S = Xn @ Xn.T; np.fill_diagonal(S, -np.inf); return S

def compute_csls_matrix(S, k=CSLS_K):
    Sf = np.where(np.isfinite(S), S, -np.inf)
    rk = np.sort(Sf, axis=1)[:, -k:].mean(axis=1)
    return 2 * S - rk[:, None] - rk[None, :]

S_csls = compute_csls_matrix(cosine_similarity_matrix(X), k=CSLS_K)

gremln_tfs = set(genes) & tfs
genie3_tfs = set(edges['regulator']) & tfs
common = gremln_tfs & genie3_tfs
print(f"GREmLN TFs: {len(gremln_tfs)} | GENIE3 regulator-TFs: {len(genie3_tfs)} | common: {len(common)}")

## 3. Ranking formulae (inlined) and rankings

`score_genes_by_seed_neighbours` (GREmLN) and the outgoing-edge-weight sum (GENIE3) are computed
for both the **seed-excluded** (primary) and **seed-inclusive** (context) views, restricted to the
common universe with dense tie-aware ranks.

In [ ]:
def score_genes_by_seed_neighbours(genes, S, seed_genes, topn=TOPN, exclude_seeds=False):
    seed_in = {g for g in seed_genes if g in set(genes)}
    G = len(genes); is_seed = np.array([g in seed_in for g in genes]); K = int(is_seed.sum())
    base = K / G if G else 0.0
    dcount = np.empty(G, int); pval = np.ones(G)
    for i in range(G):
        nn = np.argpartition(-S[i], min(topn, G - 1))[:topn]
        x = int(is_seed[nn].sum()); dcount[i] = x
        nsucc = K - int(is_seed[i])
        pval[i] = hypergeom.sf(x - 1, G - 1, nsucc, topn) if nsucc >= 0 else 1.0
    out = pd.DataFrame({"gene": genes, "gremln_csls_score": dcount,
                        "fold_enrichment": (dcount / topn) / base if base > 0 else 0.0,
                        "hypergeom_p": pval, "is_seed": is_seed})
    if exclude_seeds:
        out = out.loc[~out["is_seed"]].copy()
    return out.sort_values("gremln_csls_score", ascending=False).reset_index(drop=True)

def gremln_ranking(exclude_seeds):
    r = score_genes_by_seed_neighbours(genes, S_csls, seeds, exclude_seeds=exclude_seeds)
    t = r[r["gene"].isin(common)].copy()
    t["gremln_csls_rank"] = rankdata(-t["gremln_csls_score"], method="dense").astype(int)
    t["is_BTLA_vs_TCR_seed"] = t["gene"].isin(seeds)
    t["BTLA_vs_TCR_direction"] = t["gene"].map(lambda g: f"BTLA_{seed_dir[g]}" if g in seed_dir else "not_seed")
    t["eligible_for_fair_comparison"] = (~t["is_BTLA_vs_TCR_seed"]) if exclude_seeds else True
    return t.sort_values("gremln_csls_rank").reset_index(drop=True)

def genie3_ranking(exclude_seeds):
    st = edges[edges["target"].isin(seeds)]
    inc = (st.groupby("regulator")["weight"].agg(n_seed_targets="count", genie3_score="sum")
             .reset_index().rename(columns={"regulator": "gene"}))
    # Rank over the FULL common universe: common TFs with no outgoing edge into the BTLA seed panel
    # score 0 and tie at the bottom, mirroring GREmLN which scores every common TF (incl. zero CSLS).
    # This keeps both models on an identical candidate pool for overlap/rank-correlation.
    universe = sorted(common - set(seeds)) if exclude_seeds else sorted(common)
    inc = pd.DataFrame({"gene": universe}).merge(inc, on="gene", how="left")
    inc["n_seed_targets"] = inc["n_seed_targets"].fillna(0).astype(int)
    inc["genie3_score"] = inc["genie3_score"].fillna(0.0)
    inc = inc.sort_values("genie3_score", ascending=False).reset_index(drop=True)
    inc["genie3_rank"] = rankdata(-inc["genie3_score"], method="dense").astype(int)
    inc["is_BTLA_vs_TCR_seed"] = inc["gene"].isin(seeds)
    inc["BTLA_vs_TCR_direction"] = inc["gene"].map(lambda g: f"BTLA_{seed_dir[g]}" if g in seed_dir else "not_seed")
    return inc

rk = {}
for mode, excl in (("seed_excluded", True), ("seed_inclusive", False)):
    gr, g3 = gremln_ranking(excl), genie3_ranking(excl)
    rk[mode] = {"gr": gr, "g3": g3, "gr25": list(gr.head(25)["gene"]), "g325": list(g3.head(25)["gene"])}
    gr.to_csv(TAB / f"gremln_btla_vs_tcr_{mode}_tf_ranking.csv", index=False)
    g3.to_csv(TAB / f"genie3_btla_vs_tcr_{mode}_tf_ranking.csv", index=False)
print("seed-excluded GREmLN top10:", rk["seed_excluded"]["gr25"][:10])
print("seed-excluded GENIE3 top10:", rk["seed_excluded"]["g325"][:10])

## 4. Primary comparison (seed-excluded): overlap + rank correlation. Seed-inclusive as context.

In [ ]:
def compare(mode):
    gr, g3 = rk[mode]["gr"], rk[mode]["g3"]
    m = gr[["gene", "gremln_csls_score", "gremln_csls_rank", "is_BTLA_vs_TCR_seed"]].merge(
        g3[["gene", "genie3_score", "genie3_rank"]], on="gene", how="outer")
    m["rank_difference_gremln_minus_genie3"] = m["gremln_csls_rank"] - m["genie3_rank"]
    both = m.dropna(subset=["gremln_csls_rank", "genie3_rank"])
    rho = spearmanr(both["gremln_csls_rank"], both["genie3_rank"])[0] if len(both) >= 3 else np.nan
    gr25, g325 = set(rk[mode]["gr25"]), set(rk[mode]["g325"])
    ov = {"mode": mode, "spearman_rank_rho": float(rho), "n_common_ranked_by_both": int(len(both)),
          "shared_top25": sorted(gr25 & g325), "gremln_only_top25": sorted(gr25 - g325),
          "genie3_only_top25": sorted(g325 - gr25)}
    m.sort_values("gremln_csls_rank").to_csv(TAB / f"gremln_genie3_common_universe_rank_comparison_{mode}.csv", index=False)
    return ov

overlap = {m: compare(m) for m in ("seed_excluded", "seed_inclusive")}
prim = overlap["seed_excluded"]
print(f"PRIMARY (seed-excluded): Spearman rho={prim['spearman_rank_rho']:.3f}; "
      f"shared top25={len(prim['shared_top25'])}: {prim['shared_top25']}")
print("GREmLN-only:", prim["gremln_only_top25"])
print("GENIE3-only:", prim["genie3_only_top25"])

## 5. CD4 CRISPRi perturbational validation (primary; seed-excluded top-25)

For each model's top-25, its predicted BTLA-DEG targets (GENIE3: graph targets ∩ common seeds;
GREmLN: top-100 CSLS neighbours ∩ common seeds; identical common seed set for both) are tested
for differential expression after that TF's knockdown
(adj p < DE_PADJ). Signed agreement flags whether confirmed targets move with/against the BTLA
programme. Both scores are **unsigned**, so target movement does not establish direction.

In [ ]:
# Both models are restricted to the IDENTICAL common BTLA seed set (the seeds scorable by BOTH
# models: present in the GREmLN gene universe AND reachable as GENIE3 graph targets). This keeps the
# predicted-target denominators symmetric so neither model benefits from broader seed coverage.
graph_nodes = set(edges["regulator"]) | set(edges["target"])
common_seed_set = (set(seeds) & set(genes)) & graph_nodes
print("common BTLA seed set (scorable by both models):", len(common_seed_set), "of", len(set(seeds)))

def genie3_targets_by_tf(tf_list):
    sub = edges[edges["regulator"].isin(tf_list) & edges["target"].isin(common_seed_set)]
    return {tf: set(g["target"].unique()) for tf, g in sub.groupby("regulator", sort=False)}

def csls_seed_targets_by_tf(tf_list):
    gi = {g: i for i, g in enumerate(genes)}; seed_in = common_seed_set; G = len(genes); out = {}
    for tf in tf_list:
        if tf not in gi:
            out[tf] = set(); continue
        nn = np.argpartition(-S_csls[gi[tf]], min(TOPN, G - 1))[:TOPN]
        out[tf] = {genes[j] for j in nn} & seed_in
    return out

def crispri_validation(union_by_model, de_stats_path):
    import anndata as ad, h5py
    all_tfs = sorted({t for lst in union_by_model.values() for t in lst})
    de = ad.read_h5ad(de_stats_path, backed="r"); de_obs = de.obs.reset_index(drop=True)
    de_gene = de.var["gene_name"].astype(str).values; n_meas = len(de_gene)
    def arm(cond):
        mask = ((de_obs["culture_condition"].astype(str) == cond)
                & (de_obs["target_contrast_gene_name"].astype(str).isin(all_tfs)))
        ri = np.where(mask.values)[0]; order = np.argsort(ri); sidx = ri[order]; inv = np.argsort(order)
        with h5py.File(de_stats_path, "r") as f:
            adjp = f["layers/adj_p_value"][sidx, :][inv]; logfc = f["layers/log_fc"][sidx, :][inv]
        tfn = de_obs.loc[ri, "target_contrast_gene_name"].astype(str).values
        ot = de_obs.loc[ri, "ontarget_significant"].values
        nc = de_obs.loc[ri, "n_cells_target"].values if "n_cells_target" in de_obs else np.full(len(ri), np.nan)
        # Deterministic one-row-per-TF map: if a TF has multiple rows for this condition, prefer an
        # on-target-significant row (QC-passing), else keep the earliest index. Avoids silent last-wins.
        pmap = {}
        for i, t in enumerate(tfn):
            if t not in pmap or (not bool(ot[pmap[t]]) and bool(ot[i])):
                pmap[t] = i
        return adjp, logfc, ot, nc, pmap
    a8, l8, o8, c8, p8 = arm(STIM_PRIMARY); a48, l48, o48, _, p48 = arm(STIM_SENS)
    def de_info(pmap, adjp, tf):
        if tf not in pmap: return None
        padj = adjp[pmap[tf], :]; hit = np.isfinite(padj) & (padj < DE_PADJ)
        return set(de_gene[hit]), int(hit.sum())
    sign = {g: (1 if seed_dir.get(g) == "up" else -1) for g in seeds}
    rows = []
    for model, top in union_by_model.items():
        pred_by = csls_seed_targets_by_tf(top) if model == "GREmLN" else genie3_targets_by_tf(top)
        for tf in top:
            pred = set(pred_by.get(tf, set())); present = tf in p8
            d8, d48 = de_info(p8, a8, tf), de_info(p48, a48, tf)
            rec = {"TF": tf, "model": model, "is_BTLA_vs_TCR_seed": tf in seeds,
                   "crispri_arm_present_8hr": bool(present), "crispri_arm_present_48hr": bool(tf in p48),
                   "ontarget_kd_significant": bool(o8[p8[tf]]) if present else np.nan,
                   "n_predicted_btla_targets": len(pred)}
            for lab, di in (("8hr", d8), ("48hr", d48)):
                if di is None or not pred:
                    rec[f"n_de_genes_crispri_{lab}"] = di[1] if di else np.nan
                    rec[f"n_confirmed_{lab}"] = np.nan; rec[f"frac_confirmed_{lab}"] = np.nan
                    rec[f"hypergeom_p_{lab}"] = np.nan; rec[f"confirmed_targets_{lab}"] = ""
                    continue
                dg, nde = di; conf = pred & dg; k = len(conf)
                rec[f"n_de_genes_crispri_{lab}"] = nde; rec[f"n_confirmed_{lab}"] = k
                rec[f"frac_confirmed_{lab}"] = k / len(pred)
                rec[f"hypergeom_p_{lab}"] = float(hypergeom.sf(k - 1, n_meas, nde, len(pred))) if k > 0 else np.nan
                rec[f"confirmed_targets_{lab}"] = "|".join(sorted(conf))
            na = no = 0
            if present and pred and d8:
                lfc = dict(zip(de_gene, l8[p8[tf], :]))
                for g in (pred & d8[0]):
                    lf = lfc.get(g, np.nan)
                    if not np.isfinite(lf) or lf == 0 or g not in sign: continue
                    na += int(np.sign(lf) == np.sign(sign[g])); no += int(np.sign(lf) != np.sign(sign[g]))
            rec["n_confirmed_kd_agrees_btla"] = na; rec["n_confirmed_kd_opposes_btla"] = no
            rec["net_signed_agreement"] = (na - no) / (na + no) if (na + no) else np.nan
            rows.append(rec)
    df = pd.DataFrame(rows); df["adj_hypergeom_p_8hr"] = np.nan
    for model in df["model"].unique():
        m = df["model"] == model; p = df.loc[m, "hypergeom_p_8hr"].values; ok = np.isfinite(p)
        if ok.sum():
            adj = np.full(len(p), np.nan); adj[ok] = multipletests(p[ok], method="fdr_bh")[1]
            df.loc[m, "adj_hypergeom_p_8hr"] = adj
    def status(r):
        if not r["crispri_arm_present_8hr"]: return "not_in_screen"
        if r["ontarget_kd_significant"] == False: return "failed_ontarget_qc"
        nconf, p, net = r["n_confirmed_8hr"], r["hypergeom_p_8hr"], r["net_signed_agreement"]
        if pd.notna(nconf) and nconf >= 3 and pd.notna(net) and net <= -0.5: return "usable_contradictory"
        if pd.notna(nconf) and nconf > 0 and pd.notna(p) and p < 0.05: return "usable_supportive"
        return "usable_unsupported"
    df["validation_status"] = df.apply(status, axis=1)
    return df

union_se = {"GREmLN": rk["seed_excluded"]["gr25"], "GENIE3": rk["seed_excluded"]["g325"]}
crispri = crispri_validation(union_se, bu.artifact("de_stats"))
crispri.to_csv(TAB / "gremln_genie3_cd4_crispri_validation.csv", index=False)

_QC_FRAC = {}   # QC-passed native fractions per model, for the difference bootstrap CI in the verdict
def crispri_stats(model):
    sub = crispri[(crispri.model == model) & (crispri.TF.isin(union_se[model]))]
    pres = sub[sub.crispri_arm_present_8hr == True]
    # PRIMARY population = perturbations passing the dataset on-target-significance flag (QC-passed).
    # This matches the canonical crispri_model_summary / Table 2 / Figure 3.
    qc = pres[pres.ontarget_kd_significant == True]
    fr = qc.frac_confirmed_8hr.astype(float).dropna()
    # SENSITIVITY population = all in-screen candidates (incl. QC failures), reported separately.
    fr_all = pres.frac_confirmed_8hr.astype(float).dropna()
    rng = np.random.default_rng(RANDOM_SEED)
    b = [rng.choice(fr.values, len(fr), replace=True).mean() for _ in range(5000)] if len(fr) else [np.nan]
    _QC_FRAC[model] = fr.values
    return {"model": model, "present": int(len(pres)),
            "qc_pass": int((sub.ontarget_kd_significant == True).sum()), "n_in_primary": int(len(fr)),
            "mean_frac_8hr": round(float(fr.mean()), 4) if len(fr) else np.nan,
            "median_frac_8hr": round(float(fr.median()), 4) if len(fr) else np.nan,
            "ci_lo": round(float(np.nanquantile(b, .025)), 4), "ci_hi": round(float(np.nanquantile(b, .975)), 4),
            "mean_frac_8hr_all_screen": round(float(fr_all.mean()), 4) if len(fr_all) else np.nan,
            **{s: int((sub.validation_status == s).sum()) for s in
               ["usable_supportive", "usable_unsupported", "usable_contradictory", "failed_ontarget_qc", "not_in_screen"]}}
cr_stats = {m: crispri_stats(m) for m in ("GREmLN", "GENIE3")}
pd.DataFrame(cr_stats.values()).to_csv(TAB / "crispri_summary_by_model_seed_excluded.csv", index=False)
pd.DataFrame(cr_stats.values())

## 6. Paperclip literature (identical template; annotation only)

Structured literature evidence for the **full union of both top-25 lists** under one identical query
template (`scripts/build_paperclip_review.py`; committed to `results/paperclip/`). Because coverage
is complete, a quantitative literature comparison across the model-specific candidate sets is drawn
here. Literature is annotation, never a predictive input.

In [ ]:
union_tfs = sorted(set(union_se["GREmLN"]) | set(union_se["GENIE3"]))
pc_path = bu.repo_root() / "results" / "paperclip" / "paperclip_union_top25_review.csv"
if pc_path.exists():
    pc = pd.read_csv(pc_path)
    pc_join = pd.DataFrame({"TF": union_tfs}).merge(pc, on="TF", how="left")
    pc_join["paperclip_reviewed"] = pc_join["paperclip_reviewed"].fillna(False).astype(bool)
else:
    pc_join = pd.DataFrame({"TF": union_tfs}); pc_join["paperclip_reviewed"] = False
    pc_join["paperclip_evidence_tier"] = np.nan
pc_join.to_csv(TAB / "paperclip_union_top25.csv", index=False)
paperclip_coverage_complete = bool(len(pc_join) and pc_join["paperclip_reviewed"].all())
strong_mod = set(pc_join.loc[pc_join["paperclip_evidence_tier"].isin(["strong", "moderate"]), "TF"])
gr_only = set(union_se["GREmLN"]) - set(union_se["GENIE3"])
g3_only = set(union_se["GENIE3"]) - set(union_se["GREmLN"])
shared = set(union_se["GREmLN"]) & set(union_se["GENIE3"])
paperclip_lit = {
    "coverage_complete": paperclip_coverage_complete,
    "shared_with_strong_moderate_lit": f"{len(shared & strong_mod)}/{len(shared)}",
    "gremln_only_with_strong_moderate_lit": f"{len(gr_only & strong_mod)}/{len(gr_only)}",
    "genie3_only_with_strong_moderate_lit": f"{len(g3_only & strong_mod)}/{len(g3_only)}",
}
print(f"Paperclip coverage: {pc_join['paperclip_reviewed'].sum()}/{len(pc_join)} union TFs; "
      f"complete={paperclip_coverage_complete}")
for k, v in paperclip_lit.items():
    print(f"  {k}: {v}")

## 7. BTLA multi-omics — independent orthogonal vs derived/contextual

Independent orthogonal = protein, phosphosite, co-IP (not derivable from the expression signal
driving the models). Derived/contextual = transcript DE, TF activity, kinase activity, BIONIC/GNN
modules. Kept **separate**; not collapsed into one count.

In [ ]:
INDEP = ["protein_support", "phosphosite_support", "coip_support"]
DERIV = ["transcript_support", "tf_activity_support", "kinase_activity_support",
         "bionic_support", "early_synapse_trafficking_support"]
mo_path = bu.artifact("multiomics_summary")
union_df = pd.DataFrame({"TF": union_tfs})
union_df["is_BTLA_vs_TCR_seed"] = union_df["TF"].isin(seeds)
union_df["BTLA_vs_TCR_direction"] = union_df["TF"].map(lambda g: f"BTLA_{seed_dir[g]}" if g in seed_dir else "not_seed")
union_df["in_gremln_top25"] = union_df["TF"].isin(union_se["GREmLN"])
union_df["in_genie3_top25"] = union_df["TF"].isin(union_se["GENIE3"])

integ = union_df.copy()
if mo_path.exists():
    mo = pd.read_csv(mo_path).rename(columns={"gene": "TF"})
    cols = ["TF"] + [c for c in INDEP + DERIV + ["strongest_support_layer", "independent_support",
            "overall_multiomics_support"] if c in mo.columns]
    integ = integ.merge(mo[cols], on="TF", how="left")
    def any_support(row, layers):
        return any(str(row.get(l, "")).lower() not in ("", "nan", "none", "no", "false", "0", "0.0")
                   for l in layers if l in integ.columns)
    integ["independent_orthogonal_support"] = integ.apply(lambda r: any_support(r, INDEP), axis=1)
    integ["derived_contextual_support"] = integ.apply(lambda r: any_support(r, DERIV), axis=1)
# attach compact CRISPRi + paperclip
cr_agg = crispri.groupby("TF").agg(crispri_status=("validation_status", lambda s: ";".join(sorted(set(s)))),
        crispri_best_frac_8hr=("frac_confirmed_8hr", "max")).reset_index()
pc_cols = [c for c in ["TF", "paperclip_evidence_tier", "paperclip_primary_phenotype",
           "paperclip_direction", "paperclip_key_source", "paperclip_short_rationale",
           "paperclip_reviewed"] if c in pc_join.columns]
integ = integ.merge(cr_agg, on="TF", how="left").merge(pc_join[pc_cols], on="TF", how="left")
integ.to_csv(TAB / "gremln_genie3_top25_integrated_evidence.csv", index=False)
print("Independent orthogonal support among union TFs:",
      int(integ.get("independent_orthogonal_support", pd.Series(dtype=bool)).sum()))
integ.head(12)

## 8. Report figures (primary seed-excluded + seed-inclusive context + coverage)

In [ ]:
bu.fig_top25_comparison(rk["seed_excluded"]["gr"], rk["seed_excluded"]["g3"],
                        FIG / "fig1_top25_seed_excluded.png", seeds=seeds, mode="seed-excluded")
bu.fig_overlap(rk["seed_excluded"]["gr25"], rk["seed_excluded"]["g325"],
               FIG / "fig2_top25_overlap_seed_excluded.png", seeds=seeds, title="Seed-excluded top-25 overlap")
bu.fig_top25_comparison(rk["seed_inclusive"]["gr"], rk["seed_inclusive"]["g3"],
                        FIG / "figS_top25_seed_inclusive.png", seeds=seeds, mode="seed-inclusive")
bu.fig_crispri(crispri, FIG / "fig3_crispri_target_response.png")
bu.fig_evidence_heatmap(integ, FIG / "fig4_multiomics_heatmap.png")
n_genie3_nodes = len(set(edges['regulator']) | set(edges['target']))
bu.fig_coverage(N_MASKED_EXPR_GENES, n_genie3_nodes, len(genes),
                gremln_tfs, genie3_tfs, len(set(seeds) & common), FIG / "fig5_coverage.png")
print("figures written to", FIG)

## 9. SCENIC scope note

> SCENIC pruning of the persisted top-50k GENIE3 graph was retained as an exploratory motif-support
> sensitivity analysis (notebook 01) and was **not** used as the GREmLN prior.

## 10. Supplementary (S1) — held-out seed-TF recovery (NOT a validator)

An internal ranking-consistency diagnostic: hold out a fraction of BTLA seed-TFs and score how each
model recovers them from the remaining seeds. GREmLN's recall@25 saturates at 1.00 here, which is a
seed-clustering artefact; this is **pending a leakage/asymmetry audit** and does **not** enter the
verdict. Left unexecuted by default; enable by setting `RUN_HELDOUT = True`.

In [ ]:
RUN_HELDOUT = False
if RUN_HELDOUT:
    print("Held-out recovery is supplementary; see the development archive for the full computation.")
else:
    print("Held-out recovery skipped (supplementary; excluded from verdict).")

## 11. Restrained verdict — three-validator decision rule

A model is **superior** only if it wins CRISPRi by ≥ `CRISPRI_MARGIN` on mean functional
target-response **and** has ≥ as many `usable_supportive` TFs **and** carries ≥ corroborating
literature + multi-omics support. Otherwise: equivalence-not-tested, GENIE3-superior, or
complementary. Held-out recovery is excluded.

The primary CRISPRi metric is the **mean target-response fraction over QC-passed perturbations**
(on-target-significance flag), identical to `crispri_model_summary` / Table 2 / Figure 3. The
all-screen mean (including QC failures) is retained only as a labelled sensitivity value.

In [ ]:
gr_s, g3_s = cr_stats["GREmLN"], cr_stats["GENIE3"]
d_mean = gr_s["mean_frac_8hr"] - g3_s["mean_frac_8hr"]   # QC-passed primary metric

# bootstrap 95% CI on the between-model DIFFERENCE in QC-passed mean target-response fraction
_rng = np.random.default_rng(RANDOM_SEED)
_gr, _ge = _QC_FRAC.get("GREmLN"), _QC_FRAC.get("GENIE3")
if _gr is not None and _ge is not None and len(_gr) and len(_ge):
    _db = [_rng.choice(_gr, len(_gr), replace=True).mean() - _rng.choice(_ge, len(_ge), replace=True).mean()
           for _ in range(5000)]
    diff_ci = [round(float(np.quantile(_db, .025)), 4), round(float(np.quantile(_db, .975)), 4)]
else:
    diff_ci = [None, None]

# corroborating-evidence support among each model's method-SPECIFIC top-25 candidates:
# strong/moderate Paperclip literature + independent orthogonal multi-omics (protein/PTM/coIP).
indep_support = set(integ.loc[integ.get("independent_orthogonal_support", False) == True, "TF"]) \
    if "independent_orthogonal_support" in integ.columns else set()
corrob_support = strong_mod | indep_support      # union: a TF with both counts once
corrob = {
    "GREmLN": len(gr_only & corrob_support),
    "GENIE3": len(g3_only & corrob_support),
}
# full decision rule: CRISPRi margin win AND >= usable_supportive AND >= corroborating support
gremln_wins = ((d_mean >= CRISPRI_MARGIN)
               and (gr_s["usable_supportive"] >= g3_s["usable_supportive"])
               and (corrob["GREmLN"] >= corrob["GENIE3"]))
genie3_wins = ((-d_mean >= CRISPRI_MARGIN)
               and (g3_s["usable_supportive"] >= gr_s["usable_supportive"])
               and (corrob["GENIE3"] >= corrob["GREmLN"]))
if gremln_wins:
    verdict = "GREmLN superiority"
elif genie3_wins:
    verdict = "GENIE3 superiority"
else:
    verdict = "complementary candidate prioritisation (neither met the decision rule; equivalence not tested)"

prim = overlap["seed_excluded"]
summary = {
    "ranking_primary": "seed_excluded",
    "spearman_rho_seed_excluded": prim["spearman_rank_rho"],
    "shared_top25_seed_excluded": prim["shared_top25"],
    "gremln_only_top25": prim["gremln_only_top25"],
    "genie3_only_top25": prim["genie3_only_top25"],
    "crispri_mean_frac_8hr_qc_passed": {"GREmLN": gr_s["mean_frac_8hr"], "GENIE3": g3_s["mean_frac_8hr"],
                              "difference": round(d_mean, 4), "difference_ci95": diff_ci,
                              "margin": CRISPRI_MARGIN, "population": "QC-passed (primary)"},
    "crispri_mean_frac_8hr_all_screen_sensitivity": {"GREmLN": gr_s["mean_frac_8hr_all_screen"],
                              "GENIE3": g3_s["mean_frac_8hr_all_screen"], "population": "all in-screen (sensitivity)"},
    "crispri_usable_supportive": {"GREmLN": gr_s["usable_supportive"], "GENIE3": g3_s["usable_supportive"]},
    "corroborating_support_method_specific": corrob,
    "decision_rule": "CRISPRi margin AND >= usable_supportive AND >= corroborating (literature+independent multi-omics)",
    "paperclip_coverage_complete": paperclip_coverage_complete,
    "paperclip_literature_support": paperclip_lit,
    "held_out_recovery": "SUPPLEMENTARY — excluded from verdict",
    "verdict": verdict,
}
(REG / "benchmark_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print("\nVERDICT:", verdict)

## 12. Canonical publication-data export

Freezes the analytical inputs consumed by **notebook 04** into `results/publication_data/`, applying
the two agreed fairness fixes: **(i)** both models score against the *identical common BTLA seed set*
(GREmLN∩GENIE3 reachable), and **(ii)** every common-universe TF is explicitly classified by a
**bona-fide-TF audit** (non-sequence-specific / general-machinery factors are flagged, not silently
dropped; the primary ranking still spans the full common universe). Matched target budgets (native /
top-5 / top-10) and independent-vs-derived multi-omics are exported for the figures/tables. No model
is re-run; this is a re-scoring of frozen GREmLN embeddings and the frozen GENIE3 graph.

In [ ]:
PUB = bu.repo_root() / "results" / "publication_data"; PUB.mkdir(parents=True, exist_ok=True)

# ---- (a) bona-fide-TF classification over the common universe (annotate, do not drop) ----
NON_BONAFIDE = {
    "CYCS":   ("non_transcriptional", "cytochrome c; electron transport, not a TF"),
    "KIF22":  ("non_transcriptional", "kinesin motor protein, not a TF"),
    "SRP9":   ("non_transcriptional", "signal-recognition-particle subunit, not a TF"),
    "SMAP2":  ("non_transcriptional", "ArfGAP; not a TF"),
    "STAU2":  ("non_transcriptional", "Staufen dsRNA-binding protein; not a TF"),
    "TRAF4":  ("non_transcriptional", "TNFR-associated adaptor; not a TF"),
    "GAR1":   ("non_transcriptional", "H/ACA snoRNP component; not a TF"),
    "HMGN3":  ("chromatin_architectural", "HMGN nucleosome-binding; not sequence-specific"),
    "HIRIP3": ("chromatin_architectural", "histone-chaperone-associated; not sequence-specific"),
    "GTF2A2": ("general_machinery", "general transcription factor IIA; not sequence-specific"),
    "GTF3C2": ("general_machinery", "general transcription factor IIIC; not sequence-specific"),
    "SNAPC4": ("general_machinery", "snRNA-activating complex; not sequence-specific"),
}
def classify_tf(g):
    if g in NON_BONAFIDE:
        cls, reason = NON_BONAFIDE[g]; return cls, reason, False
    return "sequence_specific_or_regulatory_TF", "", True
audit = pd.DataFrame([
    {"TF": g, "in_gremln_universe": g in gremln_tfs, "in_genie3_universe": g in genie3_tfs,
     "is_BTLA_vs_TCR_seed": g in set(seeds), "tf_class": classify_tf(g)[0],
     "is_bona_fide_tf": classify_tf(g)[2], "classification_reason": classify_tf(g)[1]}
    for g in sorted(common)])
audit.to_csv(PUB / "candidate_universe_audit.csv", index=False)
bona_universe = set(audit.loc[audit.is_bona_fide_tf, "TF"])
print(f"common universe={len(common)} | bona-fide={len(bona_universe)} | "
      f"flagged non-bona-fide={len(common) - len(bona_universe)}")

# ---- (b) identical common BTLA seed set for BOTH models ----
graph_nodes = set(edges["regulator"]) | set(edges["target"])
gremln_seeds = set(seeds) & set(genes)
genie3_seeds = set(seeds) & graph_nodes
common_seeds = sorted(gremln_seeds & genie3_seeds)
common_seed_set = set(common_seeds)
seed_univ = pd.DataFrame([
    {"seed_gene": g, "BTLA_vs_TCR_direction": ("BTLA_" + seed_dir[g]) if g in seed_dir else "na",
     "gremln_available": g in set(genes), "genie3_reachable": g in graph_nodes,
     "in_common_seed_set": g in common_seed_set} for g in sorted(seeds)])
seed_univ.to_csv(PUB / "common_seed_universe.csv", index=False)
print(f"panel seeds={len(seeds)} | GREmLN-available={len(gremln_seeds)} | "
      f"GENIE3-reachable={len(genie3_seeds)} | identical common seed set={len(common_seeds)}")

In [ ]:
# ---- (c) re-score BOTH models over the identical common seed set (fair fix #1) ----
gi = {g: i for i, g in enumerate(genes)}
def gremln_neighbour_seeds(tf):
    # common seeds among the TF's top-100 CSLS neighbours, ranked by CSLS similarity (desc)
    if tf not in gi: return []
    row = S_csls[gi[tf]]
    nn = np.argpartition(-row, min(TOPN, len(genes) - 1))[:TOPN]
    nn = nn[np.argsort(-row[nn])]
    return [genes[j] for j in nn if genes[j] in common_seed_set]
def gremln_csls_seed_sum(tf):
    # continuous tie-break: summed CSLS similarity of the common BTLA seeds among the top-100 neighbours
    if tf not in gi: return 0.0
    row = S_csls[gi[tf]]
    nn = np.argpartition(-row, min(TOPN, len(genes) - 1))[:TOPN]
    nn = nn[np.argsort(-row[nn])]
    return float(sum(float(row[j]) for j in nn if genes[j] in common_seed_set))
g3c = edges[edges["target"].isin(common_seed_set)]
g3_sorted = {r: g.sort_values("weight", ascending=False)["target"].tolist() for r, g in g3c.groupby("regulator")}
g3_weight = g3c.groupby("regulator")["weight"].sum()
g3_count = g3c.groupby("regulator")["target"].nunique()   # linked common-BTLA-seed targets (count)
def gremln_score(tf): return len(gremln_neighbour_seeds(tf))
def genie3_score(tf): return float(g3_weight.get(tf, 0.0))
def genie3_n_seed_targets(tf): return int(g3_count.get(tf, 0))   # rankings-neutral display quantity

def build_ranking(exclude_seeds):
    rows = []
    for tf in sorted(common):
        is_seed = tf in set(seeds)
        if exclude_seeds and is_seed: continue
        rows.append({"TF": tf, "gremln_score": gremln_score(tf), "genie3_score": genie3_score(tf),
                     "gremln_csls_seed_sum": round(gremln_csls_seed_sum(tf), 6),
                     "genie3_n_seed_targets": genie3_n_seed_targets(tf),
                     "is_BTLA_vs_TCR_seed": is_seed, "is_bona_fide_tf": tf in bona_universe,
                     "BTLA_vs_TCR_direction": ("BTLA_" + seed_dir[tf]) if tf in seed_dir else "not_seed"})
    d = pd.DataFrame(rows)
    d["gremln_dense_rank"] = rankdata(-d["gremln_score"], method="dense").astype(int)
    d["genie3_dense_rank"] = rankdata(-d["genie3_score"], method="dense").astype(int)
    # dense rank is on the integer seed count; ties are ordered by the continuous CSLS seed-sum (desc)
    return d.sort_values(["gremln_dense_rank", "gremln_csls_seed_sum"], ascending=[True, False]).reset_index(drop=True)

primary = build_ranking(exclude_seeds=True)
context = build_ranking(exclude_seeds=False)
primary.to_csv(PUB / "primary_rankings_common_universe.csv", index=False)
context.to_csv(PUB / "seed_inclusive_rankings.csv", index=False)
assert not primary["is_BTLA_vs_TCR_seed"].any(), "seeds leaked into the primary seed-excluded ranking"

gr25 = list(primary.sort_values(["gremln_dense_rank", "gremln_csls_seed_sum"], ascending=[True, False]).head(25)["TF"])
g325 = list(primary.sort_values("genie3_dense_rank").head(25)["TF"])
union_primary_tfs = sorted(set(gr25) | set(g325))
up = primary[primary.TF.isin(union_primary_tfs)][["TF", "gremln_dense_rank", "genie3_dense_rank",
      "is_BTLA_vs_TCR_seed", "is_bona_fide_tf"]].copy()
up["in_gremln_top25"] = up.TF.isin(gr25); up["in_genie3_top25"] = up.TF.isin(g325)
up["status"] = np.where(up.in_gremln_top25 & up.in_genie3_top25, "shared",
               np.where(up.in_gremln_top25, "GREmLN_specific", "GENIE3_specific"))
up.sort_values(["status", "TF"]).to_csv(PUB / "top25_union_primary.csv", index=False)
rho_primary = spearmanr(primary["gremln_dense_rank"], primary["genie3_dense_rank"])[0]
print(f"primary top-25: GREmLN∩GENIE3 shared={len(set(gr25)&set(g325))} | union={len(union_primary_tfs)} | "
      f"Spearman rho={rho_primary:.3f}")
print("union not in old Paperclip review (if any):",
      sorted(set(union_primary_tfs) - set(pc_join['TF'])))

In [ ]:
# ---- (d) canonical CRISPRi with matched target budgets (native / top5 / top10) ----
def canonical_targets(tf, model):
    if model == "GREmLN": return gremln_neighbour_seeds(tf)
    return [t for t in g3_sorted.get(tf, []) if t in common_seed_set]

def canonical_crispri(union_by_model, de_stats_path, budgets=(None, 5, 10)):
    import anndata as ad, h5py
    all_tfs = sorted({t for v in union_by_model.values() for t in v})
    de = ad.read_h5ad(de_stats_path, backed="r"); de_obs = de.obs.reset_index(drop=True)
    de_gene = de.var["gene_name"].astype(str).values; n_meas = len(de_gene)
    def arm(cond):
        mask = ((de_obs["culture_condition"].astype(str) == cond)
                & (de_obs["target_contrast_gene_name"].astype(str).isin(all_tfs)))
        ri = np.where(mask.values)[0]; order = np.argsort(ri); sidx = ri[order]; inv = np.argsort(order)
        with h5py.File(de_stats_path, "r") as f:
            adjp = f["layers/adj_p_value"][sidx, :][inv]; logfc = f["layers/log_fc"][sidx, :][inv]
        tfn = de_obs.loc[ri, "target_contrast_gene_name"].astype(str).values
        ot = de_obs.loc[ri, "ontarget_significant"].values
        pmap = {}
        for i, t in enumerate(tfn):
            if t not in pmap or (not bool(ot[pmap[t]]) and bool(ot[i])): pmap[t] = i
        return adjp, logfc, ot, pmap
    a8, l8, o8, p8 = arm(STIM_PRIMARY); a48, l48, o48, p48 = arm(STIM_SENS)
    sign = {g: (1 if seed_dir.get(g) == "up" else -1) for g in seeds}
    def de_hits(pmap, adjp, tf):
        if tf not in pmap: return None
        padj = adjp[pmap[tf], :]; hit = np.isfinite(padj) & (padj < DE_PADJ)
        return set(de_gene[hit]), int(hit.sum())
    rows = []
    for model, top in union_by_model.items():
        for tf in top:
            ranked = canonical_targets(tf, model); present = tf in p8
            d8, d48 = de_hits(p8, a8, tf), de_hits(p48, a48, tf)
            r = {"TF": tf, "model": model, "is_BTLA_vs_TCR_seed": tf in set(seeds),
                 "in_screen_8hr": bool(present), "in_screen_48hr": bool(tf in p48),
                 "ontarget_qc_pass": (bool(o8[p8[tf]]) if present else np.nan),
                 "n_predicted_native": len(ranked)}
            for b in budgets:
                lab = "native" if b is None else ("top%d" % b)
                pred = set(ranked if b is None else ranked[:b]); npred = len(pred)
                r["n_predicted_" + lab] = npred
                for al, di in (("8hr", d8), ("48hr", d48)):
                    if di is None or npred == 0:
                        r["n_confirmed_%s_%s" % (lab, al)] = np.nan
                        r["frac_%s_%s" % (lab, al)] = np.nan; r["p_%s_%s" % (lab, al)] = np.nan
                    else:
                        dg, nde = di; conf = pred & dg; k = len(conf)
                        r["n_confirmed_%s_%s" % (lab, al)] = k; r["frac_%s_%s" % (lab, al)] = k / npred
                        r["p_%s_%s" % (lab, al)] = float(hypergeom.sf(k - 1, n_meas, nde, npred)) if k > 0 else np.nan
                        if al == "8hr": r["confirmed_targets_" + lab] = "|".join(sorted(conf))
            na = no = 0
            if present and d8:
                lfc = dict(zip(de_gene, l8[p8[tf], :]))
                for g in (set(ranked) & d8[0]):
                    lf = lfc.get(g, np.nan)
                    if not np.isfinite(lf) or lf == 0 or g not in sign: continue
                    na += int(np.sign(lf) == np.sign(sign[g])); no += int(np.sign(lf) != np.sign(sign[g]))
            net = (na - no) / (na + no) if (na + no) else np.nan
            r["n_kd_agrees_btla"] = na; r["n_kd_opposes_btla"] = no; r["net_signed_agreement"] = net
            r["response_direction"] = ("no_response" if not (na + no) else
                "BTLA_concordant" if net >= 0.5 else "anti_concordant" if net <= -0.5 else "mixed")
            rows.append(r)
    df = pd.DataFrame(rows)
    # keep a raw in-screen fraction (incl. QC-fail) for the all-screen sensitivity arm, then treat
    # failed-on-target-QC perturbations as NOT SCORED (NaN) so they are never counted as a zero response.
    df["frac_native_8hr_all_screen"] = df["frac_native_8hr"]
    _null = [x for x in df.columns if x.startswith(("n_confirmed_", "frac_", "p_", "confirmed_targets_"))
             and x != "frac_native_8hr_all_screen"]
    df.loc[df["ontarget_qc_pass"] == False, _null] = np.nan
    df["adj_p_native_8hr"] = np.nan
    for m in df.model.unique():
        idx = df.model == m; p = df.loc[idx, "p_native_8hr"].values; ok = np.isfinite(p)
        if ok.sum():
            adj = np.full(len(p), np.nan); adj[ok] = multipletests(p[ok], method="fdr_bh")[1]
            df.loc[idx, "adj_p_native_8hr"] = adj
    def status(r):
        if not r["in_screen_8hr"]: return "not_in_screen"
        if r["ontarget_qc_pass"] == False: return "failed_ontarget_qc"
        n, p, net = r["n_confirmed_native_8hr"], r["p_native_8hr"], r["net_signed_agreement"]
        if pd.notna(n) and n >= 3 and pd.notna(net) and net <= -0.5: return "usable_contradictory"
        if pd.notna(n) and n > 0 and pd.notna(p) and p < 0.05: return "usable_supportive"
        return "usable_unsupported"
    df["validation_status"] = df.apply(status, axis=1)
    return df

crispri_canon = canonical_crispri({"GREmLN": gr25, "GENIE3": g325}, bu.artifact("de_stats"))
crispri_canon.to_csv(PUB / "crispri_all_screen_sensitivity.csv", index=False)
qc = crispri_canon[(crispri_canon.in_screen_8hr) & (crispri_canon.ontarget_qc_pass == True)].copy()
qc.to_csv(PUB / "crispri_primary_qc_passed.csv", index=False)
budget_cols = ["TF", "model", "in_screen_8hr", "ontarget_qc_pass"]
crispri_canon[budget_cols + [c for c in crispri_canon.columns if "top5" in c]].to_csv(PUB / "crispri_matched_budget_top5.csv", index=False)
crispri_canon[budget_cols + [c for c in crispri_canon.columns if "top10" in c]].to_csv(PUB / "crispri_matched_budget_top10.csv", index=False)

def model_summary(m):
    sub = crispri_canon[crispri_canon.model == m]
    q = sub[(sub.in_screen_8hr) & (sub.ontarget_qc_pass == True)]
    fr = q["frac_native_8hr"].astype(float).dropna()
    rng = np.random.default_rng(RANDOM_SEED)
    boot = [rng.choice(fr.values, len(fr), replace=True).mean() for _ in range(5000)] if len(fr) else [np.nan]
    d = {"model": m, "present_in_screen": int(sub.in_screen_8hr.sum()),
         "qc_passed": int((sub.ontarget_qc_pass == True).sum()), "n_in_primary": int(len(q)),
         "mean_frac_native_8hr": round(float(fr.mean()), 4) if len(fr) else np.nan,
         "median_frac_native_8hr": round(float(fr.median()), 4) if len(fr) else np.nan,
         "ci_lo": round(float(np.nanquantile(boot, .025)), 4), "ci_hi": round(float(np.nanquantile(boot, .975)), 4),
         "mean_frac_top5_8hr": round(float(q["frac_top5_8hr"].astype(float).mean()), 4),
         "mean_frac_top10_8hr": round(float(q["frac_top10_8hr"].astype(float).mean()), 4),
         "mean_frac_all_screen_native_8hr": round(float(sub["frac_native_8hr_all_screen"].astype(float).mean()), 4)}
    for s in ["usable_supportive", "usable_unsupported", "usable_contradictory", "failed_ontarget_qc", "not_in_screen"]:
        d[s] = int((sub.validation_status == s).sum())
    for rd in ["BTLA_concordant", "anti_concordant", "mixed", "no_response"]:
        d["dir_" + rd] = int((q.response_direction == rd).sum())
    return d
crispri_model = pd.DataFrame([model_summary("GREmLN"), model_summary("GENIE3")])
crispri_model.to_csv(PUB / "crispri_model_summary.csv", index=False)
print(crispri_model[["model", "present_in_screen", "qc_passed", "mean_frac_native_8hr", "ci_lo", "ci_hi",
                     "mean_frac_top5_8hr", "mean_frac_top10_8hr", "usable_supportive"]].to_string(index=False))

In [ ]:
# ---- (e) evidence joins (annotation only): Paperclip + independent-vs-derived multi-omics ----
INDEP = ["protein_support", "phosphosite_support", "coip_support"]
DERIV = ["transcript_support", "tf_activity_support", "kinase_activity_support",
         "bionic_support", "early_synapse_trafficking_support"]
uni = pd.DataFrame({"TF": union_primary_tfs}).merge(
    up[["TF", "status", "gremln_dense_rank", "genie3_dense_rank", "is_BTLA_vs_TCR_seed", "is_bona_fide_tf"]],
    on="TF", how="left")
pc_full = pd.read_csv(bu.repo_root() / "results" / "paperclip" / "paperclip_union_top25_review.csv")
pc_cols = [c for c in ["TF", "paperclip_evidence_tier", "paperclip_primary_phenotype", "paperclip_direction",
           "tcr_checkpoint_evidence", "activation_exhaustion_evidence", "btla_specific_evidence",
           "paperclip_key_source", "paperclip_short_rationale", "paperclip_reviewed"] if c in pc_full.columns]
paperclip_candidate = uni.merge(pc_full[pc_cols], on="TF", how="left")
paperclip_candidate["paperclip_reviewed"] = paperclip_candidate["paperclip_reviewed"].fillna(False).astype(bool)
paperclip_candidate.to_csv(PUB / "paperclip_candidate_evidence.csv", index=False)

mo_path = bu.artifact("multiomics_summary")
mo_out = uni[["TF", "status", "is_BTLA_vs_TCR_seed"]].copy()
for col in INDEP + DERIV:
    mo_out[col] = np.nan
if mo_path.exists():
    mo = pd.read_csv(mo_path).rename(columns={"gene": "TF"})
    keep = ["TF"] + [c for c in INDEP + DERIV if c in mo.columns]
    mo_out = uni[["TF", "status", "is_BTLA_vs_TCR_seed"]].merge(mo[keep], on="TF", how="left")
# "not_significant"/"not_measured" mean tested-without-support and must NOT count as evidence
# (fixes e.g. RBPJ protein_support="not_significant" being mis-scored as independent support).
def _has(v): return str(v).lower() not in ("", "nan", "none", "no", "false", "0", "0.0",
                                           "not_significant", "not_measured", "hypothesis_only")
mo_out["independent_orthogonal_support"] = mo_out[[c for c in INDEP if c in mo_out.columns]].apply(
    lambda r: any(_has(x) for x in r), axis=1)
mo_out["derived_contextual_support"] = mo_out[[c for c in DERIV if c in mo_out.columns]].apply(
    lambda r: any(_has(x) for x in r), axis=1)
mo_out.to_csv(PUB / "multiomics_candidate_evidence.csv", index=False)

# ---- long-form per-observation evidence (full provenance) for the candidate union ----
# Tables 4A/4B and Supplementary S2 are generated from THIS, never from hand-typed labels.
lf_path = bu.artifact("multiomics_long")
LF_COLS = ["gene", "evidence_layer", "contrast", "timepoint", "condition", "direction",
           "effect_size", "adjusted_p", "tf_activity_score", "kinase_activity_score",
           "bionic_cluster", "phosphosite", "ip_condition", "source_file",
           "direct_or_inferred", "independent_or_derived", "support_level", "notes"]
if lf_path.exists():
    lf = pd.read_csv(lf_path)
    lf = lf[lf["gene"].isin(union_primary_tfs)].copy()
    for c in LF_COLS:
        if c not in lf.columns:
            lf[c] = np.nan
    lf = lf[LF_COLS].sort_values(["gene", "evidence_layer", "contrast", "timepoint"]).reset_index(drop=True)
else:
    lf = pd.DataFrame(columns=LF_COLS)
    print("WARNING: multiomics_long artifact missing:", lf_path)
lf.to_csv(PUB / "multiomics_long_evidence.csv", index=False)
print("long-form evidence rows for union candidates:", len(lf),
      "| candidates covered:", lf["gene"].nunique())

# integrated evidence map (grouped, NOT collapsed into one score)
cr8 = crispri_canon.set_index(["TF", "model"])
integ_rows = []
for _, r in uni.iterrows():
    tf = r["TF"]; model = "GREmLN" if r["status"] in ("shared", "GREmLN_specific") else "GENIE3"
    key = (tf, model) if (tf, model) in cr8.index else ((tf, "GREmLN") if (tf, "GREmLN") in cr8.index else (tf, "GENIE3") if (tf, "GENIE3") in cr8.index else None)
    cr = cr8.loc[key] if key in cr8.index else None
    pc = paperclip_candidate.set_index("TF").loc[tf]; mo = mo_out.set_index("TF").loc[tf]
    integ_rows.append({
        "TF": tf, "status": r["status"], "is_BTLA_vs_TCR_seed": r["is_BTLA_vs_TCR_seed"],
        "is_bona_fide_tf": r["is_bona_fide_tf"], "gremln_rank": r["gremln_dense_rank"], "genie3_rank": r["genie3_dense_rank"],
        "crispri_status": (cr["validation_status"] if cr is not None else "not_in_screen"),
        "ontarget_qc_pass": (cr["ontarget_qc_pass"] if cr is not None else np.nan),
        "frac_native_8hr": (cr["frac_native_8hr"] if cr is not None else np.nan),
        "response_direction": (cr["response_direction"] if cr is not None else "na"),
        "paperclip_evidence_tier": pc.get("paperclip_evidence_tier", np.nan),
        "protein_support": mo.get("protein_support", np.nan), "phosphosite_support": mo.get("phosphosite_support", np.nan),
        "coip_support": mo.get("coip_support", np.nan), "transcript_support": mo.get("transcript_support", np.nan),
        "tf_activity_support": mo.get("tf_activity_support", np.nan), "kinase_activity_support": mo.get("kinase_activity_support", np.nan),
        "bionic_support": mo.get("bionic_support", np.nan),
        "independent_orthogonal_support": mo.get("independent_orthogonal_support", False),
        "derived_contextual_support": mo.get("derived_contextual_support", False)})
candidate_integrated = pd.DataFrame(integ_rows)
candidate_integrated.to_csv(PUB / "candidate_integrated_evidence.csv", index=False)
print("evidence rows:", len(candidate_integrated), "| paperclip reviewed:",
      int(paperclip_candidate.paperclip_reviewed.sum()), "/", len(paperclip_candidate))

In [ ]:
# ---- (f) model/benchmark specification + manifest ----
n_edges = len(edges)
spec = pd.DataFrame([
    ("biological_task", "BTLA+TCR vs TCR 4h TF prioritisation (249-DEG panel)", "same for both", "identical target task"),
    ("cells", "CD4 NTC (CZI Perturb-seq)", "same cells", "same cells"),
    ("expression_input", "donor-ComBat log1p-CPM (masked)", "pre-ComBat log1p-CPM (same 50k cells)", "different numerical inputs"),
    ("preprocessing", "HVG-4000 -> CPM/log1p -> ComBat(donor)", "tokenizer rank/zero structure", "GREmLN needs raw-like structure"),
    ("graph_prior", "none (co-expression forest)", "masked GENIE3 50k-edge graph", "GREmLN is handed GENIE3's graph"),
    ("graph_size", str(n_edges) + " edges", "same graph as prior", "shared topology"),
    ("model_checkpoint", "GENIE3 (ExtraTrees)", "GREmLN model.ckpt (zero-shot)", "no fine-tuning"),
    ("gene_universe", str(len(set(edges['regulator'])|set(edges['target']))) + " graph genes", str(len(genes)) + " embedded genes", "coverage differs"),
    ("common_seed_universe", str(len(common_seeds)) + " identical common seeds", str(len(common_seeds)), "fair fix #1"),
    ("common_bona_fide_TF_universe", str(len(bona_universe)) + " of " + str(len(common)) + " common TFs", "same", "bona-fide audit"),
    ("scoring_method", "summed outgoing TF->seed edge weight", "CSLS top-100 seed-neighbour count (k=10)", "unsigned, non-comparable raw scores -> ranks"),
    ("seed_handling", "seed-excluded primary; seed-inclusive context", "same", "identical rule"),
    ("predicted_target_definition", "graph edges into common seeds", "top-100 CSLS neighbours in common seeds", "model-specific target sets"),
    ("crispri_validation_arm", "Stim8hr primary; Stim48hr sensitivity", "same", "orthogonal functional readout"),
    ("primary_fairness_limitation", "supplies GENIE3 graph to GREmLN; different expression inputs", "same", "measures embedding value given the prior, not two independent methods"),
], columns=["characteristic", "GENIE3", "GREmLN", "implication_for_comparison"])
spec.to_csv(PUB / "model_benchmark_specification.csv", index=False)

files = ["model_benchmark_specification", "common_seed_universe", "candidate_universe_audit",
         "primary_rankings_common_universe", "seed_inclusive_rankings", "top25_union_primary",
         "crispri_primary_qc_passed", "crispri_all_screen_sensitivity", "crispri_matched_budget_top5",
         "crispri_matched_budget_top10", "crispri_model_summary", "paperclip_candidate_evidence",
         "multiomics_candidate_evidence", "multiomics_long_evidence", "candidate_integrated_evidence"]
man = []
for name in files:
    p = PUB / (name + ".csv"); d = pd.read_csv(p)
    man.append({"file": name + ".csv", "rows": len(d), "cols": d.shape[1], "md5": bu.md5(p)})
manifest = pd.DataFrame(man)
manifest.to_csv(PUB / "publication_data_manifest.csv", index=False)
print(manifest.to_string(index=False))
print("\ncanonical publication-data package written to", PUB)

## 13. Audited Paperclip literature evidence (v2 — `<TF> TCR inhibition`)

Updated literature-evidence summary from the **auditable v2 pipeline**
(`scripts/run_paperclip_tcr_inhibition_audit.py`, branch `paperclip-tcr-inhibition-audit-v2`).
It re-runs Paperclip with a single fixed query per candidate (`<TF> TCR inhibition`, top-10,
PMC), builds a per-TF evidence packet, and adjudicates each candidate with a fixed LLM judge
(`claude-opus-4-8`) against a JSON schema and a deterministic rubric. Every tier traces to
supplied Paperclip result IDs; nothing is hand-typed.

This **supersedes the legacy manual review in Section 6** for literature annotation. It is
**annotation only**: it does not alter any ranking, the CRISPRi analysis, the multi-omics
joins, or the verdict. `final_usable_tier` is the judge tier when it passes rubric validation,
otherwise `missing` (flagged, never silently downgraded).

Source: `results/paperclip/v2_tcr_inhibition/` (`tf_evidence_tiers.csv`, `audit_summary.md`,
`run_manifest.json`; legacy-vs-audited diagnostic in `legacy_vs_audited_tiers.csv`).

In [ ]:
# --- 13. Audited Paperclip literature evidence (v2) — per-model summary (annotation only) ---
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

def _find_v2():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "results" / "paperclip" / "v2_tcr_inhibition"
        if cand.exists():
            return cand
    raise FileNotFoundError("results/paperclip/v2_tcr_inhibition not found")

V2 = _find_v2()
tiers = pd.read_csv(V2 / "tf_evidence_tiers.csv")
manifest = json.loads((V2 / "run_manifest.json").read_text())

TIER_ORDER = ["strong", "moderate", "weak", "none", "missing"]
for c in ("in_gremln_top25", "in_genie3_top25"):
    tiers[c] = tiers[c].astype(str).str.lower().eq("true")
tiers["rubric_valid"] = tiers["rubric_valid"].astype(str).str.lower().eq("true")

print(f"Audited Paperclip v2 — query '{manifest['exact_query_template']}', "
      f"judge {manifest['judge_model_identifier']}, {manifest['paperclip_version']}, "
      f"top_k={manifest['top_k']}")
print(f"Candidates: {len(tiers)} | unique papers: {manifest['unique_papers'] if 'unique_papers' in manifest else 'see audit_summary.md'}")
print("Annotation only — rankings, CRISPRi and the verdict are unchanged.\n")

# (a) tier distribution per model (final usable tier)
def _dist(mask, label):
    s = tiers[mask]
    row = {t: int((s["final_usable_tier"] == t).sum()) for t in TIER_ORDER}
    row["strong_or_moderate"] = int(s["final_usable_tier"].isin(["strong", "moderate"]).sum())
    row["n"] = len(s)
    return pd.Series(row, name=label)

dist = pd.DataFrame([
    _dist(tiers["in_gremln_top25"], "GREmLN top-25"),
    _dist(tiers["in_genie3_top25"], "GENIE3 top-25"),
    _dist(tiers["candidate_group"] == "shared", "shared"),
])[["n", *TIER_ORDER, "strong_or_moderate"]]
print("Evidence-tier distribution by model (final usable tier):")
display(dist)

# (b) strong / moderate candidates (annotation only)
_MODEL = {"shared": "shared", "gremln_only": "GREmLN", "genie3_only": "GENIE3"}
sm = tiers[tiers["final_usable_tier"].isin(["strong", "moderate"])].copy()
sm["model"] = sm["candidate_group"].map(_MODEL)
sm["_ord"] = sm["final_usable_tier"].map({"strong": 0, "moderate": 1})
sm = sm.sort_values(["_ord", "model", "TF"])
print("\nStrong / moderate candidates (annotation only):")
display(sm[["TF", "model", "final_usable_tier", "confidence",
            "papers_relevant", "primary_relevant_papers", "key_paper_ids"]].reset_index(drop=True))

# (c) shared top-25 candidates and their audited tiers
shared = tiers[tiers["candidate_group"] == "shared"][["TF", "final_usable_tier", "confidence"]]
print("\nShared top-25 candidates (nominated by both models):")
display(shared.sort_values("final_usable_tier").reset_index(drop=True))

# (d) rubric-flagged candidates (excluded from usable tiers; not counted as evidence)
flagged = tiers[~tiers["rubric_valid"]]
if len(flagged):
    print("\nRubric-flagged (final_usable_tier = missing; not counted as evidence):")
    display(flagged[["TF", "judge_tier", "final_usable_tier", "rubric_reasons"]].reset_index(drop=True))